--------------------------------------

# _1. Preprocessing_

The current notebook focus mainly on the preprocessing step. 

The goal is to prepare and export two versions of the dataset

- a semi-prepared and a 

- fully prepared version. 
    
Both version will constitute the baseline for the remaining tasks    

### _1.1. Load (raw) Data_

In [20]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
%matplotlib inline

In [2]:
raw_dataset = "data/new_godash_dataset.csv"
raw_df = pd.read_csv(raw_dataset, index_col=0)

### _1.2. Rename some columns_

In [3]:
all_features = ['Seg_id', 'Arr_time', 'Del_Time', 'Stall_Dur', 'Rep_Level', 'Del_Rate',
                'Act_Rate', 'Byte_Size', 'Buff_Level', 'Seg_Dur', 'Codec',
                'Width', 'Height', 'FPS', 'Play_Pos', 'RTT', 'Protocol', 'P.1203',
                'Clae', 'Duanmu', 'Yin', 'Yu', 'algo',  'nb_nodes', 'anomaly', 'level', 'tag']

features = all_features[:-3]
target = all_features[-1]


raw_df = raw_df.rename(columns={'Seg_#': 'Seg_id'})
raw_df.head(5)

,Seg_id,Arr_time,Del_Time,Stall_Dur,Rep_Level,Del_Rate,Act_Rate,Byte_Size,Buff_Level,Algorithm,...,P.1203,Clae,Duanmu,Yin,Yu,algo,tag,anomaly,level,nb_nodes
0,1,68.0,68.0,0.0,239.0,250.0,8.0,2132.0,2000.0,arbiter,...,1.871,0.000,51.077,-5760.485,0.240,arbiter,normal40,normal,0,84
1,2,1000.0,114.0,0.0,239.0,5785.0,329.0,82445.0,4000.0,arbiter,...,1.871,0.480,46.477,-11520.970,0.240,arbiter,normal40,normal,0,84
2,3,2114.0,210.0,0.0,572.0,10058.0,1056.0,264029.0,4887.0,arbiter,...,2.152,0.329,47.487,718.545,0.351,arbiter,normal40,normal,0,84
3,4,3361.0,340.0,0.0,1077.0,11440.0,1944.0,486242.0,5641.0,arbiter,...,2.384,0.251,48.820,1291.034,0.532,arbiter,normal40,normal,0,84
4,5,4562.0,338.0,0.0,1801.0,11537.0,1949.0,487454.0,6440.0,arbiter,...,2.621,0.170,50.668,2368.956,0.786,arbiter,normal40,normal,0,84


In [4]:
print (f"Dataset Shape : {raw_df.shape}")
print (f"Rows with nan values :  ({raw_df.isna().sum().sum()})")
print (f"\n\nList of {len (raw_df.columns)} columns : \n{raw_df.columns}")

Dataset Shape : (85706, 28)
Rows with nan values :  (2096)


List of 28 columns : 
Index(['Seg_id', 'Arr_time', 'Del_Time', 'Stall_Dur', 'Rep_Level', 'Del_Rate',
       'Act_Rate', 'Byte_Size', 'Buff_Level', 'Algorithm', 'Seg_Dur', 'Codec',
       'Width', 'Height', 'FPS', 'Play_Pos', 'RTT', 'Protocol', 'P.1203',
       'Clae', 'Duanmu', 'Yin', 'Yu', 'algo', 'tag', 'anomaly', 'level',
       'nb_nodes'],
      dtype='object')


### _1.3. Clean_

In [5]:
raw_df = raw_df.drop('Algorithm', axis=1)
raw_df["algo"].value_counts(normalize=True)

elastic         0.167760
conventional    0.167410
bba             0.167270
arbiter         0.166733
exponential     0.165636
logistic        0.165193
Name: algo, dtype: float64

In [6]:
raw_df['tag'].value_counts()

normal40       3600
corrupt5       3600
reorder40      3600
duplicate20    3600
normal20       3600
normal30       3600
corrupt10      3600
duplicate5     3600
normal10       3600
duplicate10    3600
reorder5       3600
duplicate50    3600
corrupt20      3600
normal50       3600
reorder10      3600
duplicate30    3600
corrupt30      3600
reorder50      3600
duplicate40    3600
reorder30      3600
reorder20      3600
normal5        3600
corrupt40      3534
corrupt50      2912
normal0          60
Name: tag, dtype: int64

#### _nan values_

In [7]:
raw_df.isna().sum()

Seg_id          0
Arr_time        2
Del_Time        2
Stall_Dur      56
Rep_Level      83
Del_Rate       83
Act_Rate      110
Byte_Size     110
Buff_Level    110
Seg_Dur       110
Codec         110
Width         110
Height        110
FPS           110
Play_Pos      110
RTT           110
Protocol      110
P.1203        110
Clae          110
Duanmu        110
Yin           110
Yu            110
algo            0
tag             0
anomaly         0
level           0
nb_nodes        0
dtype: int64

In [8]:
raw_df = raw_df.dropna()
print (f"Rows with nan values :  ({raw_df.isna().sum().sum()})")

Rows with nan values :  (0)


### _1.4. LabelEncoding_

In [9]:
train = raw_df.copy()

In [10]:
nunique = train.nunique()
types = train.dtypes

categorical_columns = []
categorical_dims =  {}
for col in train.columns:
    if types[col] == 'object' or nunique[col] < 200:
        print(col, train[col].nunique())
        l_enc = LabelEncoder()
        train[col] = train[col].fillna("VV_likely")
        train[col] = l_enc.fit_transform(train[col].values)
        categorical_columns.append(col)
        categorical_dims[col] = len(l_enc.classes_)

Seg_id 60
Arr_time 66635
Del_Time 9437
Stall_Dur 9151
Rep_Level 16
Del_Rate 21714
Seg_Dur 1
Codec 1
Width 6
Height 6
FPS 1
Play_Pos 60
Protocol 1
algo 6
tag 25
anomaly 4
level 7
nb_nodes 10


In [11]:
# features_to_encode = ["Mode", "Algorithm", "tag"]
features_to_encode = ["algo", "Codec", "Protocol"]
for feature in features_to_encode :
    print ("==========",feature, "==========")
    print (raw_df[feature].value_counts(normalize=True))
    print ("")

========== algo ==========
elastic         0.167928
conventional    0.167484
bba             0.167298
arbiter         0.166702
exponential     0.165510
logistic        0.165078
Name: algo, dtype: float64

========== Codec ==========
h264    1.0
Name: Codec, dtype: float64

========== Protocol ==========
HTTP/1.1    1.0
Name: Protocol, dtype: float64



In [14]:
for feature in features_to_encode :
    le = LabelEncoder()
    logic = le.fit_transform(raw_df[feature])
    raw_df[feature] = logic

for feature in features_to_encode :
    print ("==========",feature, "==========")
    print (raw_df[feature].value_counts())
    print ("")

========== algo ==========
3    14374
2    14336
1    14320
0    14269
4    14167
5    14130
Name: algo, dtype: int64

========== Codec ==========
0    85596
Name: Codec, dtype: int64

========== Protocol ==========
0    85596
Name: Protocol, dtype: int64



### _1.5. Zero or near-zero variance_

The `sklearn.feature_selection.VarianceThreshold` transformer will by default remove all zero-variance features. 

We can also pass a threshold as an argument to make it remove features whose variance is lower than the threshold [[ref](https://neptune.ai/blog/feature-selection-methods)].

In [15]:
raw_X = raw_df[features]
constant_filter = VarianceThreshold(threshold=0)
constant_filter.fit(raw_X)
features_zero_var = [feature for feature in raw_X.columns \
                     if feature not in list(raw_X.columns[constant_filter.get_support()])]
print ("Features with zero Variance : ", features_zero_var)

Features with zero Variance :  ['Seg_Dur', 'Codec', 'FPS', 'Protocol']


Let us delete it next :

In [16]:
raw_df = raw_df.drop(features_zero_var, axis=1)
raw_df.tag.unique()

array(['normal40', 'normal50', 'normal5', 'reorder20', 'reorder30',
       'corrupt50', 'corrupt40', 'duplicate40', 'reorder50', 'corrupt30',
       'duplicate30', 'reorder10', 'corrupt20', 'corrupt5', 'duplicate50',
       'reorder5', 'duplicate10', 'normal10', 'duplicate5', 'corrupt10',
       'normal30', 'normal20', 'duplicate20', 'reorder40', 'normal0'],
      dtype=object)

### _1.6. Features engineering_

Add the following features, relaxing the target feature _**tag**_ :

> **_MOS_** : feature (0 to 5) quantisizing the QoS and based on Yin model
> 
> **_tag2_** : two classes _normal, abnormal_ destined to binary classification
> 
> **_anomaly_** : 4 classes _normal, reorder, duplicate_ and _corrupt_ -> destined to less complex classifiers
> 
> **_tag50_** : 4 classes _normal, reorder(50), duplicate(50)_ and _corrupt(50)_ -> destined to less complex classifiers

In [17]:
# tag2
new_target = [raw_df["anomaly"] != "normal"]
raw_df["tag2"] = new_target[0]

No need to precise level in "normal" cases

In [18]:
raw_df.loc[raw_df["anomaly"]=="normal", 'tag'] = 'normal'
raw_df.tag.value_counts()

normal         21660
corrupt20       3600
duplicate20     3600
corrupt10       3600
duplicate5      3600
duplicate10     3600
reorder5        3600
duplicate50     3600
corrupt5        3600
reorder10       3600
reorder20       3600
duplicate30     3600
corrupt30       3600
reorder50       3600
duplicate40     3600
reorder30       3600
reorder40       3600
corrupt40       3525
corrupt50       2811
Name: tag, dtype: int64

##### _Add MOS column_
recall 

<br>
<div style="float: left;">
<img src="https://www.researchgate.net/profile/Huang-Nan-Huang/publication/272562658/figure/tbl1/AS:667598513520646@1536179297362/Mean-opinion-score-MOS.png" alt="drawing" width="400"/>
</div>
</br>


In [21]:
x = raw_df['Yin'].values.reshape(-1, 1) #returns a numpy array
min_max_scaler = MinMaxScaler()
raw_df['mos'] = min_max_scaler.fit_transform(x)*5

labelz = ["bad", "poor", "fair", "good", "excelent"]
raw_df["mos_label"] = pd.cut(raw_df.mos, bins=5, labels=labelz, right=False)

raw_df["mos_label"].value_counts()

bad         85373
poor          177
fair           36
excelent        6
good            4
Name: mos_label, dtype: int64

In [22]:
# label encode the MOS column
di = {'bad': 1, 'poor': 2, 'fair': 3, 'good': 4, 'excelent': 5}
raw_df["MOS"] = raw_df.mos_label.replace(di)
raw_df.MOS.value_counts()

1    85373
2      177
3       36
5        6
4        4
Name: MOS, dtype: int64

### _1.7. Save a semi-cleaned version_

This version will be further processed bellow, let us save a copy as it is.

In [23]:
raw_df.to_csv("data/semi_new_godash_dataset.csv")
cleaned_dataset = "data/semi_new_godash_dataset.csv"
pd.read_csv(cleaned_dataset, index_col=0).head(5)

,Seg_id,Arr_time,Del_Time,Stall_Dur,Rep_Level,Del_Rate,Act_Rate,Byte_Size,Buff_Level,Width,...,Yu,algo,tag,anomaly,level,nb_nodes,tag2,mos,mos_label,MOS
0,1,68.0,68.0,0.0,239.0,250.0,8.0,2132.0,2000.0,320.0,...,0.240,0,normal,normal,0,84,False,0.012588,bad,1
1,2,1000.0,114.0,0.0,239.0,5785.0,329.0,82445.0,4000.0,320.0,...,0.240,0,normal,normal,0,84,False,0.000000,bad,1
2,3,2114.0,210.0,0.0,572.0,10058.0,1056.0,264029.0,4887.0,512.0,...,0.351,0,normal,normal,0,84,False,0.026746,bad,1
3,4,3361.0,340.0,0.0,1077.0,11440.0,1944.0,486242.0,5641.0,640.0,...,0.532,0,normal,normal,0,84,False,0.027997,bad,1
4,5,4562.0,338.0,0.0,1801.0,11537.0,1949.0,487454.0,6440.0,736.0,...,0.786,0,normal,normal,0,84,False,0.030353,bad,1


----------------------------------------

## _Version II_

### _1.8. Standardizing using  StandardScaler_

- This consists on standardizing the features to have zero mean and unit variance. 

- This step ensures that each feature is on the same scale and prevents features with larger magnitudes from dominating the analysis.

- Done using _z-score_ standardization implemented using Scikit-learn's _**StandardScaler()**_


In [24]:
# Recall all these have been deleted --> ['Seg_Dur', 'Codec', 'FPS', 'Protocol']
all_features = ['Seg_id', 'Arr_time', 'Del_Time', 'Stall_Dur', 'Rep_Level', 'Del_Rate',
                'Act_Rate', 'Byte_Size', 'Buff_Level', 'Width',
                'Height', 'Play_Pos', 'RTT','P.1203', 'Clae',
                'Duanmu', 'Yin', 'Yu', 'algo', 'nb_nodes',
                'mos', 'mos_label', 'MOS', 'anomaly', 'level', 'tag2', 'tag']

feature_to_encode = ['anomaly', 'tag', 'tag2']
target = "tag"

unused_feat = ["mos_label"] 

In [25]:
features_to_scale = [ col for col in all_features if col not in unused_feat+feature_to_encode] 
scalar = StandardScaler() 
to_scale_df = raw_df[features_to_scale]

# Standardizing 
scalar.fit(to_scale_df) 
scaled_features_data = scalar.transform(to_scale_df) 
scaled_features_data.shape

(85596, 23)

In [26]:
scaled_df = pd.DataFrame(scaled_features_data, columns= features_to_scale)
scaled_df.shape

(85596, 23)

In [27]:
scaled_df.loc[:,['anomaly', 'tag', 'tag2']] = raw_df.loc[:,['anomaly', 'tag', 'tag2']].reset_index()
scaled_df.shape

(85596, 26)

### _1.9. Label Encoding "labels"_

##### "Anomaly"

In [28]:
custom_mapping_anomaly = {'normal': 0, 'reorder': 1, 'duplicate': 2, 'corrupt': 3}
    
le_anomaly = LabelEncoder()
le_anomaly.classes_ = list(custom_mapping_anomaly.keys())
scaled_df["anomaly"] = le_anomaly.transform(scaled_df["anomaly"])
print (le_anomaly.classes_)

['normal', 'reorder', 'duplicate', 'corrupt']


In [29]:
scaled_df.tag.value_counts()

normal         21660
corrupt20       3600
duplicate20     3600
corrupt10       3600
duplicate5      3600
duplicate10     3600
reorder5        3600
duplicate50     3600
corrupt5        3600
reorder10       3600
reorder20       3600
duplicate30     3600
corrupt30       3600
reorder50       3600
duplicate40     3600
reorder30       3600
reorder40       3600
corrupt40       3525
corrupt50       2811
Name: tag, dtype: int64

##### "tag"

In [30]:
from sklearn.preprocessing import LabelEncoder
custom_mapping_tag = { 
                          'normal': 0,
                      'duplicate5': 1, 'duplicate10': 2, 'duplicate20': 3, 
                     'duplicate30': 4, 'duplicate40': 5, 'duplicate50': 6,   
                       'reorder5':  7,  'reorder10':  8,   'reorder20':  9, 
                      'reorder30': 10,  'reorder40': 11,   'reorder50': 12,                       
                       'corrupt5': 13,  'corrupt10': 14,   'corrupt20': 15, 
                      'corrupt30': 16,  'corrupt40': 17,   'corrupt50': 18, 
                 }
    
le_tag = LabelEncoder()
le_tag.classes_ = list(custom_mapping_tag.keys())
scaled_df["tag"] = le_tag.transform(scaled_df["tag"])
print (le_tag.classes_)

['normal', 'duplicate5', 'duplicate10', 'duplicate20', 'duplicate30', 'duplicate40', 'duplicate50', 'reorder5', 'reorder10', 'reorder20', 'reorder30', 'reorder40', 'reorder50', 'corrupt5', 'corrupt10', 'corrupt20', 'corrupt30', 'corrupt40', 'corrupt50']


In [31]:
scaled_df["mos_label"] = raw_df["mos_label"].values
scaled_df.columns

Index(['Seg_id', 'Arr_time', 'Del_Time', 'Stall_Dur', 'Rep_Level', 'Del_Rate',
       'Act_Rate', 'Byte_Size', 'Buff_Level', 'Width', 'Height', 'Play_Pos',
       'RTT', 'P.1203', 'Clae', 'Duanmu', 'Yin', 'Yu', 'algo', 'nb_nodes',
       'mos', 'MOS', 'level', 'anomaly', 'tag', 'tag2', 'mos_label'],
      dtype='object')

### _1.10. Reorganize columns_

In [32]:
# Recall, all the following have been deleted --> ['Seg_Dur', 'Codec', 'FPS', 'Protocol']
all_features = ['Seg_id', 'Arr_time', 'Del_Time', 'Stall_Dur', 'Rep_Level', 'Del_Rate',
                'Act_Rate', 'Byte_Size', 'Buff_Level', 'Width',
                'Height', 'Play_Pos', 'RTT', 'P.1203', 'Clae',
                'Duanmu', 'Yin', 'Yu', 'algo', 'nb_nodes', 'mos', 'MOS', 'mos_label', 
                'anomaly', 'level', 'tag2', 'tag']
scaled_df = scaled_df [all_features]

### _1.11. Export the new scalled/(fully) cleaned version of the dataset_

In [33]:
scaled_df.to_csv("data/scaled_new_godash_dataset.csv")
scaled_df = pd.read_csv("data/scaled_new_godash_dataset.csv", index_col=0)
scaled_df.head(5)

,Seg_id,Arr_time,Del_Time,Stall_Dur,Rep_Level,Del_Rate,Act_Rate,Byte_Size,Buff_Level,Width,...,Yu,algo,nb_nodes,mos,MOS,mos_label,anomaly,level,tag2,tag
0,-1.697807,-0.445033,-0.204334,-0.164204,-1.294465,-1.160341,-1.160370,-1.160361,-0.833377,-1.247338,...,0.360347,-1.462647,0.524565,-1.043858,-0.045817,bad,0,-1.083746,False,0
1,-1.640070,-0.441006,-0.198458,-0.164204,-1.294465,-0.396335,-1.008632,-1.008502,-0.575036,-1.247338,...,0.360347,-1.462647,0.524565,-1.114816,-0.045817,bad,0,-1.083746,False,0
2,-1.582333,-0.436192,-0.186196,-0.164204,-1.108483,0.193475,-0.664977,-0.665158,-0.460461,-0.978685,...,0.367256,-1.462647,0.524565,-0.964049,-0.045817,bad,0,-1.083746,False,0
3,-1.524596,-0.430804,-0.169590,-0.164204,-0.826439,0.384235,-0.245217,-0.244990,-0.363067,-0.799582,...,0.378520,-1.462647,0.524565,-0.956997,-0.045817,bad,0,-1.083746,False,0
4,-1.466860,-0.425614,-0.169846,-0.164204,-0.422082,0.397624,-0.242853,-0.242699,-0.259859,-0.665255,...,0.394329,-1.462647,0.524565,-0.943719,-0.045817,bad,0,-1.083746,False,0


In [34]:
scaled_df.isna().sum().sum()

0